# 🗺️ GeoAttribute-LLM: Geospatial Evaluation & Error Analysis
**Automated Map Quality Control, Ground-Truth Benchmarking & Spatial Error Taxonomies**

This notebook provides an interactive exploratory environment for LLM-powered road attribute extraction.
- **Dataset**: 50 curated OpenStreetMap (OSM) segments from Cluj-Napoca, Romania.
- **Orchestration**: Prefect workflow with cached tasks and retries.
- **Evaluation**: Scikit-learn Precision, Recall, and Spatial Error Taxonomies.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load benchmark metrics and dataset
metrics_path = Path('../reports/metrics_summary.json')
data_path = Path('../data/osm_ground_truth_cluj.json')

with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

with open(data_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

df = pd.DataFrame([{
    'segment_id': item['segment_id'],
    'name': item['name'],
    'road_type': item['expected_attributes']['road_type'],
    'speed_limit': item['expected_attributes']['speed_limit_kmh'],
    'barrier_present': item['expected_attributes']['barrier_present'],
    'barrier_type': item['expected_attributes']['barrier_type'],
    'surface': item['expected_attributes']['surface'],
    'oneway': item['expected_attributes']['oneway'],
    'anomaly': item['expected_attributes']['anomaly_flag']
} for item in dataset])

print(f"Loaded {len(df)} road segments across {df['road_type'].nunique()} road classifications.")
df.head()

## 1. Quantitative Performance per Road Attribute

In [ ]:
summary_rows = []
for attr_name, vals in metrics['attributes'].items():
    row = {'Attribute': attr_name}
    row.update({k: f"{v*100:.1f}%" if isinstance(v, float) else v for k, v in vals.items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

## 2. Precision & Recall by Road Attribute

In [ ]:
attr = metrics['attributes']
categories = ['Road Type', 'Barrier', 'Surface', 'One-Way']
p_vals = [attr['road_type']['precision'] * 100, attr['barrier_present']['precision'] * 100, attr['surface_type']['precision'] * 100, attr['oneway']['precision'] * 100]
r_vals = [attr['road_type']['recall'] * 100, attr['barrier_present']['recall'] * 100, attr['surface_type']['recall'] * 100, attr['oneway']['recall'] * 100]

fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, p_vals, width, label='Precision (%)', color='#0284c7')
ax.bar(x + width/2, r_vals, width, label='Recall (%)', color='#0f766e')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Attribute Extraction Precision & Recall', fontweight='bold', pad=10)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontweight='bold')
ax.set_ylim(80, 105)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Spatial Error Taxonomy Analysis
Discrepancies mapped to domain error categories to guide prompt iteration.

In [ ]:
err_data = metrics.get('error_taxonomy_breakdown', {})
if err_data:
    err_df = pd.DataFrame(list(err_data.items()), columns=['Error Taxonomy', 'Count'])
    fig, ax = plt.subplots(figsize=(6, 3), dpi=120)
    ax.barh(err_df['Error Taxonomy'], err_df['Count'], color='#e11d48')
    ax.set_xlabel('Count of Occurrences', fontweight='bold')
    ax.set_title('Spatial Error Taxonomy Distribution', fontweight='bold', pad=10)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No errors detected in current evaluation run!')